<a href="https://colab.research.google.com/github/francji1/01RAD/blob/main/HW/01RAD_Ex02_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01RAD - Homework 02: Regression to the mean (Galton's heights)

In 1886 Francis Galton compared the heights of 1,078 fathers and their adult sons and noticed that sons of tall fathers tend to be shorter than their fathers, and sons of short fathers taller. He called the phenomenon *regression towards mediocrity*; the name stuck to the whole method (Lecture 1, history). The data are the `fsdata.csv` file from Exercise 01 (heights in inches).

Model of Lecture 1: $Y_i = \beta_0 + \beta_1 x_i + e_i$ with $x_i$ the father's height and $Y_i$ the son's height, independent errors $e_i \sim (0, \sigma^2)$. Tools of Lecture 2 (slides pp. 14–32 of the lecture PDF): standard errors $\widehat{\sigma}(\widehat{\beta}_j) = s_n \sqrt{\delta_j}$, **confidence intervals** (Czech: intervaly spolehlivosti) $\widehat{\beta}_j \pm t_{1-\alpha/2}(n-2)\,\widehat{\sigma}(\widehat{\beta}_j)$ and the **tests** (testy hypotéz) with statistics $T_j = (\widehat{\beta}_j - b)/\widehat{\sigma}(\widehat{\beta}_j)$ compared with $t_{1-\alpha/2}(n-2)$.

## Submission
Work through the tasks in this notebook (code plus short written answers in markdown cells; Python or R, as you prefer). Submit through GitHub **before the next exercise**: fork `francji1/01RAD`, save the notebook into the `HW/` folder as `01RAD_Ex02_HW_<Surname>.ipynb`, with your surname in place of `<Surname>` and without the brackets, for example `01RAD_Ex02_HW_Novak.ipynb` (team: surnames joined by an underscore), and open a pull request to `main`. The pull request should add your one file only; do not modify or delete other files. A selected student solution, and a reference solution if needed, will be published in `HW/` a week later.

Useful calls: `stats.t.ppf(1 - alpha/2, df)` for the $t$ quantile, `stats.t.sf(abs(T), df)` and `stats.t.cdf(T, df)` for tail probabilities, `model.conf_int(alpha)`, `model.t_test('father = 1')`, `model.predict(pd.DataFrame({'father': [...]}))`, `np.corrcoef`, `df.sample(n, random_state=...)`.

## Data

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', palette='deep')

url_fsdata = 'https://raw.githubusercontent.com/francji1/01RAD/main/data/fsdata.csv'
fsdata = pd.read_csv(url_fsdata, index_col=0)
print(f'Rows: {fsdata.shape[0]}, columns: {fsdata.shape[1]}')
fsdata.describe().T

Rows: 1078, columns: 2


,count,mean,std,min,25%,50%,75%,max
father,1078.0,67.687097,2.744868,59.00800,65.787735,67.76660,69.60298,75.43393
son,1078.0,68.684070,2.814702,58.50708,66.931232,68.61582,70.46597,78.36479


## Task 01 - Fit and recap
Fit the regression of `son` on `father`. Compute $\widehat{\beta}_0$, $\widehat{\beta}_1$, $s_n$ and both standard errors by hand (formulas of Lectures 1 and 2) and check them against `statsmodels` (R: `summary(lm(son ~ father))`). Interpret $\widehat{\beta}_1$ in one sentence.

In [4]:
# your solution
x = fsdata['father']
y = fsdata['son']
n = len(fsdata)

x_mean = x.mean()
y_mean = y.mean()
S_xx = ((x-x_mean)**2).sum()
S_xy = ((x-x_mean)*(y-y_mean)).sum()
b_1 = S_xy/S_xx
b_0 = y_mean - b_1*x_mean
e = y - (b_0 + b_1*x)
e_2 = (e**2).sum()
s_n = np.sqrt(e_2/(n-2))

sigma_b_1 = s_n * np.sqrt(1/S_xx)
sigma_b_0 = s_n * np.sqrt((1/n) + (x_mean**2/S_xx))

print(f'b0 = {b_0:.4f}, b1 = {b_1:.4f}, s_n = {s_n:.4f}, '
      f'Směrodatná chyba sigma(b0) = {sigma_b_0:.4f}, Směrodatná chyba sigma(b1) = {sigma_b_1:.4f}')

model = smf.ols('son ~ father', data=fsdata).fit()
print(model.summary())
print('statsmodels s_n =', np.sqrt(model.scale))

b0 = 33.8866, b1 = 0.5141, s_n = 2.4366, Směrodatná chyba sigma(b0) = 1.8324, Směrodatná chyba sigma(b1) = 0.0270
                            OLS Regression Results                            
Dep. Variable:                    son   R-squared:                       0.251
Model:                            OLS   Adj. R-squared:                  0.251
Method:                 Least Squares   F-statistic:                     361.2
Date:                Sun, 11 Oct 2026   Prob (F-statistic):           1.12e-69
Time:                        13:27:59   Log-Likelihood:                -2488.7
No. Observations:                1078   AIC:                             4981.
Df Residuals:                    1076   BIC:                             4991.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------

Ruční výpočet se shoduje s výstupem statsmodels. Vzhledem k tomu, že syn je vysvětlovaná veličina, tak sklon $\beta_1 = 0.51$ naznačuje, že pokud se otcové liší o 1 palec tak jejich synové se budou lišit o půl palce.

## Task 02 - Confidence intervals
Compute the 95 % and the 99 % confidence intervals for $\beta_0$ and $\beta_1$ by hand, using the $t$ quantile, and compare with `conf_int` (R: `confint`). Why is the 99 % interval wider? Compare the widths of the intervals for $\beta_0$ and $\beta_1$ in absolute terms and relative to the estimates: which parameter is estimated more precisely? Relate your answer to the remark in Lecture 2.

In [5]:
# your solution
for alpha in [0.05, 0.01]:
    t_q = stats.t.ppf(1 - alpha / 2, df=n - 2)
    print(f"t_{1- alpha/2}({n - 2}) = {t_q:.4f}   (normal quantile: {stats.norm.ppf(1 - alpha/2):.4f})")

    ci_manual = pd.DataFrame({
        'lower': [b_0 - t_q * sigma_b_0, b_1 - t_q * sigma_b_1],
        'upper': [b_0 + t_q * sigma_b_0, b_1 + t_q * sigma_b_1],
    }, index=['Intercept', 'father'])
    ci_manual['width'] = ci_manual['upper'] - ci_manual['lower']
    ci_manual['rel_width'] = ci_manual['width'] / [b_0, b_1]

    print("\nmanual", (1-alpha)*100, "% confidence intervals:")
    print(ci_manual)
    print("\nstatsmodels conf_int:")
    print(model.conf_int(alpha))

t_0.975(1076) = 1.9622   (normal quantile: 1.9600)

manual 95.0 % confidence intervals:
               lower      upper     width  rel_width
Intercept  30.291213  37.481996  7.190784   0.212201
father      0.461019   0.567167  0.106148   0.206477

statsmodels conf_int:
                   0          1
Intercept  30.291213  37.481996
father      0.461019   0.567167
t_0.995(1076) = 2.5804   (normal quantile: 2.5758)

manual 99.0 % confidence intervals:
               lower      upper     width  rel_width
Intercept  29.158387  38.614822  9.456434   0.279061
father      0.444296   0.583890  0.139593   0.271533

statsmodels conf_int:
                   0          1
Intercept  29.158387  38.614822
father      0.444296   0.583890


Ručně vypočtené intervaly se shodují s výstupem conf_int.
Interval je pro $99$% širší, protože chceme být jistější, že skutečnou hodnotu obsahuje.
Absolutně je interval pro $\beta_0$ mnohem širší (7,19 oproti 0,106), ale to je dáno hlavně tím, že $\beta_0$ je mnohem větší číslo. Relativně k odhadu jsou oba intervaly podobně široké (zhruba 0.21), sklon je odhadnut nepatrně přesněji.


## Task 03 - Is the father's height informative?
Test $H_0: \beta_1 = 0$ against $H_1: \beta_1 \neq 0$ at $\alpha = 0.05$: report $T_1$, the critical value $t_{0.975}(n-2)$, the $p$-value and the decision. What does the rejection say about the relationship, and what does it *not* say (think of the lecture's warning about linearity)?

In [8]:
# your solution
alpha = 0.05

T1 = (b_1 - 0) / sigma_b_1
t_crit = stats.t.ppf(1 - alpha/2, df=n - 2)
p_value = 2 * stats.t.sf(abs(T1), df=n - 2)

print(f'T1 = {T1:.4f}')
print(f'kritická hodnota t_0.975({n-2}) = {t_crit:.4f}')
print(f'p-hodnota = {p_value:.3e}')
print('Rozhodnutí:', 'zamítáme H0' if abs(T1) > t_crit else 'nezamítáme H0')

print(model.t_test('father = 0'))

T1 = 19.0062
kritická hodnota t_0.975(1076) = 1.9622
p-hodnota = 1.121e-69
Rozhodnutí: zamítáme H0
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.5141      0.027     19.006      0.000       0.461       0.567


$T_1 = 19.0062$, kritická hodnota $t_{0,975}(1076) = 1{.}962$, $p = 1.12 *10^{-69}$. Protože $|T_1| > t_{0,975}$, hypotézu $H_0: \beta_1 = 0$ na hladině významnosti 5 % zamítáme. To odpovídá i tomu, že 95% interval pro $\beta_1$ neobsahuje nulu.

Zamítnutí říká, že výška otce nese statisticky významnou informaci o výšce syna (sklon je nenulový a kladný). Neříká ale, že vztah je lineární, test předpokládá přímkový model a správnost tohoto předpokladu nijak neověřuje, to je potřeba posoudit např. z grafu dat nebo reziduí. Neříká ani, že vztah je silný.

## Task 04 - Regression to the mean
If sons were on average exactly as tall as their fathers plus a constant, the slope would be $\beta_1 = 1$. Test $H_0: \beta_1 = 1$ against $H_1: \beta_1 \neq 1$ (the test with a given candidate value $b_1$ from Lecture 2) and also against the **one-sided alternative** (jednostranná alternativa) $H_1: \beta_1 < 1$: the lecture treats two-sided alternatives, for the one-sided one use the same statistic $T = (\widehat{\beta}_1 - 1)/\widehat{\sigma}(\widehat{\beta}_1)$, reject when $T < -t_{1-\alpha}(n-2)$, and the $p$-value is $P(t(n-2) < T)$. Check the duality of the two-sided test with the 95 % confidence interval from Task 02. Explain in your own words why Galton spoke of *regression* (regrese k průměru) and what the estimated slope says about sons of very tall and very short fathers.

In [9]:
# your solution
alpha = 0.05
b_test = 1
T = (b_1 - b_test) / sigma_b_1

# oboustranný test
t_crit_2 = stats.t.ppf(1 - alpha/2, df=n - 2)
p_2 = 2 * stats.t.sf(abs(T), df=n - 2)

# jednostranný test
t_crit_1 = stats.t.ppf(1 - alpha, df=n - 2)
p_1 = stats.t.cdf(T, df=n - 2)

print(f'T = {T:.4f}')
print(f'Oboustranný:  kritická hodnota ±{t_crit_2:.4f}, p = {p_2:.3e}, '
      f'{"zamítáme H0" if abs(T) > t_crit_2 else "nezamítáme H0"}')
print(f'Jednostranný: kritická hodnota -{t_crit_1:.4f}, p = {p_1:.3e}, '
      f'{"zamítáme H0" if T < -t_crit_1 else "nezamítáme H0"}')

print(model.t_test('father = 1'))

lo, hi = model.conf_int(0.05).loc['father']
print(f'95% interval pro beta1: [{lo:.4f}, {hi:.4f}], obsahuje 1? {lo <= 1 <= hi}')

for f in [60, 76]:
    s = b_0 + b_1 * f
    print(f'otec {f}: předpovězený syn {s:.2f}, rozdíl syn - otec = {s - f:+.2f}')

T = -17.9641
Oboustranný:  kritická hodnota ±1.9622, p = 2.610e-63, zamítáme H0
Jednostranný: kritická hodnota -1.6463, p = 1.305e-63, zamítáme H0
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.5141      0.027    -17.964      0.000       0.461       0.567
95% interval pro beta1: [0.4610, 0.5672], obsahuje 1? False
otec 60: předpovězený syn 64.73, rozdíl syn - otec = +4.73
otec 76: předpovězený syn 72.96, rozdíl syn - otec = -3.04


$T =  -17{,}96$.

Oboustranný test ($H_1: \beta_1 \neq 1$): $|T| > t_{0,975}(1076) \approx 1{,}962$, $p \approx 2{,}6\cdot10^{-63}$, $H_0$ zamítáme.
Jednostranný test ($H_1: \beta_1 < 1$): $T < -t_{0,95}(1076) \approx -1{,}646$, $p = P(t(1076) < T) \approx 1{,}3\cdot10^{-63}$, $H_0$ zamítáme.

Dualita: 95% interval pro $\beta_1$ z Tasku 02, $[0{,}461;\,0{,}567]$, neobsahuje hodnotu 1, což odpovídá zamítnutí oboustranného testu na hladině 5 %.

Regrese k průměru: Odhadnutý sklon $\approx 0{,}51$ znamená, že synové v průměru převezmou jen asi polovinu odchylky otce od průměru. Synové velmi vysokých otců jsou proto v průměru nižší než jejich otcové a synové velmi nízkých otců vyšší. Výšky synů se tedy „vracejí“ k průměru populace, odtud Galtonův název regrese. Platí to pro průměr skupiny synů, ne pro každého jednotlivce, a variabilita výšek v populaci se tím nezmenšuje.

## Task 05 - The intercept
Test $H_0: \beta_0 = 0$. Then write $\widehat{\beta}_0 = \overline{y} - \widehat{\beta}_1 \overline{x}$, plug in the sample means and explain why the intercept is far from zero even though sons are on average only about one inch taller than their fathers. Would a model through the origin make sense here?

In [10]:
# your solution

T0 = (b_0 - 0) / sigma_b_0
t_crit = stats.t.ppf(0.975, df=n - 2)
p0 = 2 * stats.t.sf(abs(T0), df=n - 2)
print(f'T0 = {T0:.4f}, kritická hodnota = {t_crit:.4f}, p = {p0:.3e}, '
      f'{"zamítáme H0" if abs(T0) > t_crit else "nezamítáme H0"}')
print(model.t_test('Intercept = 0'))


print(f'\nx_bar = {x_mean:.4f}, y_bar = {y_mean:.4f}, y_bar - x_bar = {y_mean - x_mean:.4f}')
print(f'b1 * x_bar = {b_1 * x_mean:.4f}')
print(f'b0 = y_bar - b1*x_bar = {y_mean:.4f} - {b_1 * x_mean:.4f} = {y_mean - b_1 * x_mean:.4f}')
print(f'kdyby b1 = 1: b0 = {y_mean - 1 * x_mean:.4f}')

model0 = smf.ols('son ~ father - 1', data=fsdata).fit()
print(f'\nmodel přes počátek: sklon = {model0.params["father"]:.4f}, '
      f's_n = {np.sqrt(model0.scale):.4f}')
print(f'model s interceptem: sklon = {b_1:.4f}, s_n = {s_n:.4f}')

T0 = 18.4935, kritická hodnota = 1.9622, p = 1.604e-66, zamítáme H0
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            33.8866      1.832     18.493      0.000      30.291      37.482

x_bar = 67.6871, y_bar = 68.6841, y_bar - x_bar = 0.9970
b1 * x_bar = 34.7975
b0 = y_bar - b1*x_bar = 68.6841 - 34.7975 = 33.8866
kdyby b1 = 1: b0 = 0.9970

model přes počátek: sklon = 1.0139, s_n = 2.7958
model s interceptem: sklon = 0.5141, s_n = 2.4366


Test: $T_0 \approx 18{,}5$, $p \approx 10^{-66}$, takže $H_0: \beta_0 = 0$ na hladině 5 % zamítáme (95% interval $[30{,}3;\,37{,}5]$ také neobsahuje nulu).

Proč je intercept velký: $\widehat\beta_0 = \bar y - \widehat\beta_1\bar x = 68{,}68 - 0{,}514\cdot 67{,}69 \approx 68{,}68 - 34{,}80 = 33{,}89$. Synové jsou sice v průměru jen asi o 1 palec vyšší ($\bar y - \bar x \approx 1$), ale sklon je pouze asi 0,51, takže přímka „započítá“ jen polovinu výšky otce a zbytek musí doplnit intercept, aby přímka prošla bodem průměrů. Kdyby byl sklon 1, intercept by byl asi 1. Velký intercept je tedy důsledkem regrese k průměru. Navíc odpovídá výšce otce 0 palců, tedy hodnotě daleko mimo data, a nemá proto přímou interpretaci.

Model přes počátek zde smysl nedává: hypotézu $\beta_0 = 0$ jsme zamítli a vynucení průchodu bodem $(0,0)$, který je daleko od dat, by zkreslilo sklon (vyjde asi 1,01, takže by model chybně „popřel“ regresi k průměru) a zhoršilo přizpůsobení ($s_n$ vzroste z 2,44 na 2,80).

## Task 06 - Point predictions
Predict the mean height of sons whose fathers are 60, 64, 68, 72 and 76 inches tall and tabulate the difference *predicted son minus father*. For which father's height do the two coincide? (Confidence and prediction intervals for these predictions come in Lecture 4; here point predictions are enough.)

In [14]:
# your solution
fathers = pd.DataFrame({'father': [60, 64, 68, 72, 76]})
fathers['predicted_son'] = model.predict(fathers)
fathers['son_minus_father'] = fathers['predicted_son'] - fathers['father']
print(fathers.round(3))

x_star = b_0 / (1 - b_1)
print(f'\nPředpověď se rovná výšce otce pro x* = {x_star:.3f} palce')



   father  predicted_son  son_minus_father
0      60         64.732             4.732
1      64         66.789             2.789
2      68         68.845             0.845
3      72         70.901            -1.099
4      76         72.958            -3.042

Předpověď se rovná výšce otce pro x* = 69.739 palce


Malí otcové mají v průměru vyšší syny, než jsou sami (otec 60 palců → syn asi o 4,7 palce vyšší). Vysocí otcové mají v průměru nižší syny (otec 76 palců → syn asi o 3 palce nižší). Syn je stejně vysoký jako otec, když otec měří asi 69,7 palce. Synové se tedy přibližují k průměru, což je regrese k průměru.

## Task 07 - Reverse regression
Fit the reverse regression of `father` on `son` and report its slope. Verify numerically that the product of the two slopes equals the square of the sample correlation coefficient $r$ of the two heights (`np.corrcoef`), and explain why both slopes are smaller than 1. Is regression to the mean a causal effect of fathers on sons, or a statistical property of the two distributions?

In [15]:
# your solution

model_rev = smf.ols('father ~ son', data=fsdata).fit()
b1_rev = model_rev.params['son']
print(model_rev.params)

r = np.corrcoef(fsdata['father'], fsdata['son'])[0, 1]
print(f'\nsklon son~father  = {b_1:.4f}')
print(f'sklon father~son  = {b1_rev:.4f}')
print(f'součin sklonů     = {b_1 * b1_rev:.6f}')
print(f'r = {r:.4f}, r^2  = {r**2:.6f}')
print(f'1 / sklon son~father = {1/b_1:.4f}  (není to obrácená regrese!)')

s_x, s_y = fsdata['father'].std(), fsdata['son'].std()
print(f'\ns_x = {s_x:.4f}, s_y = {s_y:.4f}')
print(f'r * s_y/s_x = {r * s_y / s_x:.4f}  (= sklon son~father)')
print(f'r * s_x/s_y = {r * s_x / s_y:.4f}  (= sklon father~son)')

Intercept    34.107453
son           0.488900
dtype: float64

sklon son~father  = 0.5141
sklon father~son  = 0.4889
součin sklonů     = 0.251340
r = 0.5013, r^2  = 0.251340
1 / sklon son~father = 1.9452  (není to obrácená regrese!)

s_x = 2.7449, s_y = 2.8147
r * s_y/s_x = 0.5141  (= sklon son~father)
r * s_x/s_y = 0.4889  (= sklon father~son)


Obrácená regrese father ~ son má sklon $\approx 0{,}489$. Součin obou sklonů je $0{,}514\cdot0{,}489 \approx 0{,}2513 = r^2$ ($r \approx 0{,}501$), což plyne z vyjádření $\widehat\beta_{y\sim x} = r\,s_y/s_x$ a $\widehat\beta_{x\sim y} = r\,s_x/s_y$.

Oba sklony jsou menší než 1, protože výšky otců a synů mají podobný rozptyl ($s_x \approx 2{,}74$, $s_y \approx 2{,}81$) a korelace mezi nimi je nedokonalá ($r \approx 0{,}5 < 1$).

Regrese k průměru proto není příčinný vliv otců na syny, stejný jev nastává i v obráceném směru, a syn přitom výšku otce ovlivnit nemůže. Jde o statistickou vlastnost dvou nedokonale korelovaných veličin, extrémní hodnoty jsou zčásti dány náhodnou složkou, která se na příbuzného nepřenáší, takže jeho hodnota leží v průměru blíž středu rozdělení. Rozptyl výšek v populaci se tím nemění.

## Task 08 - Sample size and precision
Draw random subsamples of 30, 100 and 300 father-son pairs (fix the random seed), fit the model in each and compare the widths of the 95 % intervals for $\beta_1$ with the full-sample interval. Explain the pattern through $\mathrm{Var}(\widehat{\beta}_1) = \sigma^2 / S_{xx}$. *Optional:* repeat the subsampling with $n = 100$ five hundred times, treat the full-sample $\widehat{\beta}_1$ as the true value and estimate how often the 95 % interval covers it.

In [25]:
# your solution
rows = []
for size in [30, 100, 300, n]:
    sub = fsdata if size == n else fsdata.sample(size, random_state=42)
    m = smf.ols('son ~ father', data=sub).fit()
    lo, hi = m.conf_int(0.05).loc['father']
    Sxx_sub = ((sub['father'] - sub['father'].mean())**2).sum()
    rows.append({'n': size,
                 'b1': m.params['father'],
                 'lower': lo, 'upper': hi, 'width': hi - lo,
                 's_n': np.sqrt(m.scale),
                 'Sxx': Sxx_sub})

res = pd.DataFrame(rows)
res['width / width_full'] = res['width'] / res['width'].iloc[-1]
res['sqrt(n_full / n)'] = np.sqrt(n / res['n'])
print(res.round(4))

      n      b1   lower   upper   width     s_n        Sxx  \
0    30  0.6240  0.3167  0.9312  0.6145  2.1109   198.0586   
1   100  0.5146  0.3452  0.6840  0.3388  2.2537   697.0393   
2   300  0.4958  0.3925  0.5992  0.2067  2.3691  2034.8302   
3  1078  0.5141  0.4610  0.5672  0.1061  2.4366  8114.4439   

   width / width_full  sqrt(n_full / n)  
0              5.7890            5.9944  
1              3.1918            3.2833  
2              1.9474            1.8956  
3              1.0000            1.0000  


S rostoucí velikostí vzorku se 95% interval pro $\beta_1$ zužuje: šířka je 0,615 pro $n=30$, 0,339 pro $n=100$, 0,207 pro $n=300$ a 0,106 pro celý vzorek ($n=1078$). Vysvětluje to vztah $\mathrm{Var}(\widehat\beta_1) = \sigma^2/S_{xx}$, odhad $s_n$ rozptylu kolem přímky se s $n$ téměř nemění (2,1–2,4), zatímco $S_{xx} = \sum(x_i-\bar x)^2 \approx (n-1)s_x^2$ roste přibližně lineárně s $n$ (198 → 8114). Směrodatná chyba, a tím i šířka intervalu, proto klesá. Malé vzorky navíc dávají víc rozkolísané odhady (pro $n=30$ vyšlo $\widehat\beta_1 \approx 0{,}62$), všechny intervaly však hodnotu z celého vzorku obsahují.